# 03 - Data Preprocessing & Leakage Prevention

## Overview
This notebook implements the data preprocessing pipeline for the Telco Customer Churn dataset.

It defines the reusable `prepare_data(df)` function that cleans, encodes, splits, imputes, and scales the dataset while strictly enforcing **data leakage prevention best practices**.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Load raw dataset
data_path = os.path.join("..", "data", "Telco-Customer-Churn.csv") if os.path.exists(os.path.join("..", "data", "Telco-Customer-Churn.csv")) else os.path.join("data", "Telco-Customer-Churn.csv")
df = pd.read_csv(data_path)
print(f"Raw dataset shape: {df.shape}")

Raw dataset shape: (7043, 21)


In [2]:
def prepare_data(df: pd.DataFrame, test_size: float = 0.2, random_state: int = 42):
    """
    Preprocesses Telco Customer Churn DataFrame in strict order:
    1. Drop customerID.
    2. Map target Churn Yes -> 1, No -> 0.
    3. Convert SeniorCitizen to string/categorical.
    4. Coerce TotalCharges to numeric.
    5. One-hot encode categorical features with drop_first=True.
    6. Split into train/test (80/20, stratify=y, random_state=42) BEFORE imputation/scaling.
    7. Fit SimpleImputer(strategy='median') on X_train ONLY.
    8. Fit StandardScaler on X_train ONLY.
    """
    data = df.copy()
    
    # 1. Drop customerID
    if 'customerID' in data.columns:
        data = data.drop(columns=['customerID'])
        
    # 2. Map target: Churn Yes -> 1, No -> 0
    data['Churn'] = data['Churn'].map({'Yes': 1, 'No': 0})
    y = data['Churn']
    X = data.drop(columns=['Churn'])
    
    # 3. Convert SeniorCitizen to string/categorical before encoding
    if 'SeniorCitizen' in X.columns:
        X['SeniorCitizen'] = X['SeniorCitizen'].astype(str)
        
    # 4. Coerce TotalCharges to numeric
    if 'TotalCharges' in X.columns:
        X['TotalCharges'] = pd.to_numeric(X['TotalCharges'], errors='coerce')
        
    # 5. One-hot encode all categorical columns with drop_first=True
    X_encoded = pd.get_dummies(X, drop_first=True)
    
    # 6. Split into train/test BEFORE any scaling or imputation
    X_train, X_test, y_train, y_test = train_test_split(
        X_encoded, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    # 7. Fit SimpleImputer(strategy='median') on X_train ONLY
    imputer = SimpleImputer(strategy='median')
    X_train_imp = X_train.copy()
    X_test_imp = X_test.copy()
    
    X_train_imp[['TotalCharges']] = imputer.fit_transform(X_train[['TotalCharges']])
    X_test_imp[['TotalCharges']] = imputer.transform(X_test[['TotalCharges']])
    
    # 8. Fit StandardScaler on X_train ONLY
    scaler = StandardScaler()
    X_train_scaled_arr = scaler.fit_transform(X_train_imp)
    X_test_scaled_arr = scaler.transform(X_test_imp)
    
    X_train_scaled = pd.DataFrame(X_train_scaled_arr, columns=X_train_imp.columns, index=X_train_imp.index)
    X_test_scaled = pd.DataFrame(X_test_scaled_arr, columns=X_test_imp.columns, index=X_test_imp.index)
    
    return X_train_imp, X_test_imp, y_train, y_test, X_train_scaled, X_test_scaled, imputer, scaler

# Execute prepare_data pipeline
X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled, imputer, scaler = prepare_data(df)

In [3]:
# Verification 1: Train & Test Dataset Dimensions
print("=== Dataset Split Dimensions ===")
print(f"X_train shape: {X_train.shape} (80%)")
print(f"X_test shape:  {X_test.shape} (20%)")
print(f"Number of encoded features: {X_train.shape[1]}")

=== Dataset Split Dimensions ===
X_train shape: (5634, 30) (80%)
X_test shape:  (1409, 30) (20%)
Number of encoded features: 30


In [4]:
# Verification 2: Class Balance & Stratification
train_counts = y_train.value_counts()
train_pct = y_train.value_counts(normalize=True) * 100

test_counts = y_test.value_counts()
test_pct = y_test.value_counts(normalize=True) * 100

balance_df = pd.DataFrame({
    'Train Count': train_counts,
    'Train %': train_pct.round(2),
    'Test Count': test_counts,
    'Test %': test_pct.round(2)
})
print("=== Stratified Class Balance (y_train vs y_test) ===")
print(balance_df)
print(f"\nTrain Churn Rate: {y_train.mean()*100:.2f}%")
print(f"Test Churn Rate:  {y_test.mean()*100:.2f}%")

=== Stratified Class Balance (y_train vs y_test) ===
       Train Count  Train %  Test Count  Test %
Churn                                          
0             4139    73.46        1035   73.46
1             1495    26.54         374   26.54

Train Churn Rate: 26.54%
Test Churn Rate:  26.54%


In [5]:
# Verification 3: Imputation Metadata Check
train_median = imputer.statistics_[0]
print("=== Imputation Summary ===")
print(f"TotalCharges Median (computed on X_train only): ${train_median:.2f}")
print(f"Missing TotalCharges in X_train after imputation: {X_train['TotalCharges'].isnull().sum()}")
print(f"Missing TotalCharges in X_test after imputation:  {X_test['TotalCharges'].isnull().sum()}")

=== Imputation Summary ===
TotalCharges Median (computed on X_train only): $1398.12
Missing TotalCharges in X_train after imputation: 0
Missing TotalCharges in X_test after imputation:  0


In [6]:
# Verification 4: Unscaled vs Scaled Feature Preview
print("=== Unscaled X_train Sample (First 3 Rows) ===")
display(X_train[['tenure', 'MonthlyCharges', 'TotalCharges']].head(3))

print("\n=== Scaled X_train_scaled Sample (First 3 Rows) ===")
display(X_train_scaled[['tenure', 'MonthlyCharges', 'TotalCharges']].head(3))

=== Unscaled X_train Sample (First 3 Rows) ===


,tenure,MonthlyCharges,TotalCharges
3738,35,49.20,1701.65
3151,15,75.10,1151.55
4860,13,40.55,590.35



=== Scaled X_train_scaled Sample (First 3 Rows) ===


,tenure,MonthlyCharges,TotalCharges
3738,0.102371,-0.521976,-0.263290
3151,-0.711743,0.337478,-0.504815
4860,-0.793155,-0.809013,-0.751214


## Why Splitting Before Imputation/Scaling Prevents Data Leakage

### What is Data Leakage?
**Data leakage** occurs when information from outside the training dataset (specifically from the test set or validation set) is inadvertently used to fit preprocessing transformations or train machine learning models.

### 1. Imputation Leakage
- If we compute summary statistics (such as the median of `TotalCharges`) on the **entire dataset (7,043 rows)** prior to splitting, the computed median incorporates values from the test set.
- When imputing missing values in the training set using a full-dataset median, the training data absorbs information about the test set's distribution. This creates an artificially optimistic evaluation during testing.
- **Correct Approach**: Fit `SimpleImputer(strategy='median')` on `X_train` strictly (**5,634 rows**). The resulting median (**$1,398.13**) is then applied (`.transform()`) to both `X_train` and `X_test` without recomputing or inspecting `X_test` values.

### 2. Scaling Leakage
- `StandardScaler` calculates the feature mean ($\mu$) and standard deviation ($\sigma$).
- If standard scaling is fitted on the full dataset, the mean and variance of the test set bleed into the training set scaling transformation.
- **Correct Approach**: Fit `StandardScaler` on `X_train` only (`scaler.fit_transform(X_train)`), and transform `X_test` using `scaler.transform(X_test)`.

### 3. Stratified Splitting (`stratify=y`)
- Splitting prior to transformation ensures that class proportions (**26.54% Churn = Yes**) are perfectly preserved in both the training set (4,139 No / 1,495 Yes) and test set (1,035 No / 374 Yes), preventing distribution shift between splits.

> **Note on Resampling (SMOTE)**: SMOTE is intentionally excluded from this module. Resampling must only be performed inside cross-validation folds (via imbalanced-learn `Pipeline`) to avoid leaking synthetic test samples.